<img src=../figures/Brown_logo.svg width=50%>

## Data-Driven Design & Analyses of Structures & Materials (3dasm)

## Lecture 19.1

### Miguel A. Bessa | <a href = "mailto: miguel_bessa@brown.edu">miguel_bessa@brown.edu</a>  | Associate Professor

### Elvis Aguero | <a href = "mailto: elvis_vera@brown.edu">elvis_vera@brown.edu</a>  | PhD candidate


**What:** A lecture of the "3dasm" course

**Where:** This notebook comes from this [repository](https://github.com/bessagroup/3dasm_course)

**Reference for entire course:** Murphy, Kevin P. *Probabilistic machine learning: an
introduction*. MIT press, 2022. Available online [here](https://probml.github.io/pml-book/book1.html)

**How:** We try to follow Murphy's book closely, but the sequence of Chapters and Sections is
different. The intention is to use notebooks as an introduction to the topic and Murphy's book
as a resource.
* If working offline: Go through this notebook and read the book.
* If attending class in person: listen to me (!) but also go through the notebook in your laptop at the same time. Read the book.
* If attending lectures remotely: listen to me (!) via Zoom and (ideally) use two screens where you have the notebook open in 1 screen and you see the lectures on the other. Read the book.

This is the first of two lectures on **adda**. Today: how you build such a framework, and what
we learned building it. Lecture 19.2: what happened when we pointed it at a real problem.

## **OPTION 1**. Run this notebook **locally in your computer**:
1. Confirm that you have the '3dasm' mamba (or conda) environment (see Lecture 1).
2. Go to the 3dasm_course folder in your computer and pull the last updates of the [repository](https://github.com/bessagroup/3dasm_course):
```
git pull
```
    - Note: if you can't pull the repo due to conflicts (and you can't handle these conflicts), use this command (with **caution**!) and your repo becomes the same as the one online:
```
git reset --hard origin/main
```
3. Open command window and load jupyter notebook (it will open in your internet browser):
```
jupyter notebook
```
5. Open notebook of this Lecture and choose the '3dasm' kernel.

## **OPTION 2**. Use **Google's Colab** (no installation required, but times out if idle):

1. go to https://colab.research.google.com
2. login
3. File > Open notebook
4. click on Github (no need to login or authorize anything)
5. paste the git link: https://github.com/bessagroup/3dasm_course
6. click search and then click on the notebook for this Lecture.

In [1]:
# Basic plotting tools needed in Python.

import matplotlib.pyplot as plt # import plotting tools to create figures
import numpy as np # import numpy to handle a lot of things!

%config InlineBackend.figure_format = "retina" # render higher resolution images in the notebook
plt.rcParams["figure.figsize"] = (8,4) # rescale figure size appropriately for slides

# To limit the number of rows to show in a dataframe, for presentation purposes:
import pandas as pd

pd.set_option('display.max_rows', 10)

In [2]:
# In Google Colab you need to install f3dasm first (locally it is already in the '3dasm'
# environment). Uncomment the line below if you are running in Colab:

# %pip install f3dasm

from f3dasm import ExperimentData   # the same object you used in Lectures 17, 18 and 19

## Outline for today

* What an agent is: tools, nodes, a graph
* Why more than one node: safety, specialization, efficiency
* Where the state lives
* What admits a claim
* Getting started, and one lesson from building it

**Reading material**: this notebook + the
[a3dasm documentation](https://elvis-aguero.github.io/a3dasm/).

## Agents, tools, and jobs


* An LLM with a set of tools, and a job is an **agent** — a **subagent** when another agent spawns it. As of 2026 they are dispatched automatically by the coding agents you already use:

<img src=../figures/agent_products.svg width=86%>

* A fixed code path that calls a model at each step is a **workflow**. We will see how to **orchestrate** multiple of them.

## Six multi-agent architectures

There are multiple ways to arrange multiple agents to collaborate. Some people are trying Group chats, Supervisors, hierarchies, and free-form networks.

A graph architecture where each node is a possibly different agent, and edges dictate who can talk to who.

<p align="center"><img src=../figures/agentic_patterns.png width=58%></p>

<sub>Taxonomy from the LangGraph multi-agent documentation; figure from *Agentic patterns: architectures for coordinated AI systems*, Medium.</sub>

We will show you the decisions involved when architecting a _graph_ of agents.

## Why a graph of agents

Three principles: Safety, specialization, efficiency. 

We will see that there are a few advantages of arranging agents in a graph format.

## Safety I: independence needs a fresh context

An agent asked to check its own work usually agrees with it.

A second agent, started fresh and shown only the result, gives you a real second opinion.

## Safety II: restricted tools per node

Each node gets a specialized set of tools tailored to their goals.

## Specialization, one model per node

Different jobs, different models. A strong model plans, a cheaper one executes.

A fine-tuned open-weights model already is on par with frontier models on narrow tasks at a fraction of the compute. Each node can get its own backend with their own settings.

## Efficiency through parallel, cancellable work

Work is handed over in units you can cancel, retry, or abandon. A six-hour run that dies at hour five does not start again.

Independent units also run at the same time.

## When one chat is enough

Short task. Cheap to check. You already know the plan.

Then a single chat is the right tool, and a graph is overhead.

## The shared state is a file on disk

A node is stateless between calls.

Agent frameworks typically checkpoint the whole conversation, so a thread can be resumed.
We chose to make the record the state instead: that buys reproducibility rather than
resumable dialogue. (what does it mean for the record to be the state?)

In [3]:
# A record from a real run: 538 evaluations of a lattice design.
ledger = ExperimentData.from_file('.')
design, result = ledger.to_pandas()
record = design.join(result)

# the design columns are your final project's variables; the _ columns are the stamps
record[["ratio_a", "ratio_pitch", "coilable", "sigma_crit",
        "_delegation_id", "_source", "_wall_ms"]].head(6)

evaluations on the record: 538
stamped by the framework: ['_delegation_id', '_source', '_ts', '_wall_ms']


## Choosing your own topology

Ours has one node that plans and delegates to the others. That is an accident of our problem,
not a principle: it is a bottleneck and a single point of failure.

Your problem, your graph.

## Popper's advice

As of 2026, LLMs are generalists by default.

That means they tend to output reasonable sounding answers, that tend to regress to the mean, and the whole user-agent interaction is probabilistic by nature. 

We try to circumvent that by emphasizing the rules of science to all agents in the workflow. A whole deal of research has been done in the lines of philosophy of science, and we might start by delimiting the Popperian rules of science. 


## Science rule 1: one falsifiable claim

One file, quoted verbatim into every node that judges a claim. §1:

> A hypothesis is ONE falsifiable claim carrying a registered prediction — the observable
> whose occurrence would refute the claim.

Register what would refute you, before you look.


## Science rule 2: corroboration, not proof

§5:

> SUPPORTED [...] means the hypothesis survived at least one adequate attempt to refute
> it. You never "confirm" a hypothesis; you only fail to falsify it.

Four statuses, and only four: `OPEN`, `SUPPORTED`, `FALSIFIED`, `INCONCLUSIVE`.


## The reproduction gate

The deliverable is a notebook. Its code cells recompute the result from the record.

It is executed in a clean sandbox before the run may close.


## The study folder

```
my_study/
  PROBLEM_STATEMENT.md   # required: the brief
  config.yaml            # optional: model, budget, how a design is scored
  workspace/
    evaluator.py         # optional: your ground truth
```

## The problem statement file

```markdown
# Minimise a 2-D quadratic

## Objective
Minimise y = (x1 - 1)^2 + (x2 + 2)^2.

## Design space
| variable | type | bounds | units |
|---|---|---|---|
| x1 | continuous | [-5, 5] | dimensionless |
| x2 | continuous | [-5, 5] | dimensionless |
```

Objective, bounds, units. That table is a `Domain`, written in prose.

## Model, budget, and evaluator

```yaml
model: claude-haiku-4-5-20251001
eval_budget: 200
evaluator:
  entrypoint: "workspace/evaluator.py:evaluate"
  output_names: [y]
```

```python
def evaluate(x1: float, x2: float) -> float:
    return (x1 - 1.0) ** 2 + (x2 + 2.0) ** 2
```

## One call, one notebook

```python
from a3dasm import AgenticRun
report = AgenticRun(study_dir="my_study").execute()
```

Back comes `pipeline.ipynb`, the record of every evaluation, and whether the run passed
its gate.

## An agent's account of a failure

Every node writes a retrospective when a run closes: what it found hard, what blocked it, where it
contradicted itself. It is the highest-signal artifact we have, and it can still be wrong about
mechanism.

One run blamed a silent crash for a 63% evaluation failure rate. The logs showed the licence
server saturating at 16-way concurrency.

In [4]:
# What the run reported, against what the record shows.
d = design.join(result)
attempted = d[d.coilable == 1]              # a solve was only ever run on these
failed = ~attempted.riks_converged.astype(bool)

print("solves that never converged:", int(failed.sum()), "of", len(attempted))
print("failure rate:", round(100 * failed.mean(), 1), "%")
print("median wall time, failed vs converged:",
      int(attempted.loc[failed, '_wall_ms'].median() / 1000), "s vs",
      int(attempted.loc[~failed, '_wall_ms'].median() / 1000), "s")

solves that never converged: 154 of 469
failure rate: 32.8 %
median wall time, failed vs converged: 329 s vs 66 s


The retrospective is a lead. The record is the diagnosis: these solves ran five times longer than
the ones that worked before they died, which is a wait, not a crash.

## Summary

* An agentic workflow is a graph of nodes, each a model with tools, choosing its own next step.
* Split the work for **safety**, **specialization** and **efficiency**, and not otherwise.
* The state is the data, not the conversation.
* A claim is admitted by surviving refutation and by reproducing from the record.
* One folder in, one notebook out.

## Next lecture

The same framework, pointed at a real design problem, and what it actually found.

### See you next class

Have fun!